# Flash Attention Explained

**Stage 2: Memory Optimization - Notebook 20**

This notebook explores Flash Attention, one of the most impactful innovations in LLM inference. We'll cover:
- The O(n²) memory problem in standard attention
- GPU memory hierarchy (HBM vs SRAM)
- Flash Attention's tiling strategy
- O(n²) → O(n) memory complexity
- Speed improvements (1.5-2x faster)
- Memory savings (2-4x reduction)
- Installing and using flash-attn library
- Integration with real models
- Flash Attention v2 and v3 improvements

**References**:
- LLM_INFERENCE_ROADMAP.md - Stage 2
- Notebook 55: KV Cache Optimization
- Paper: "FlashAttention: Fast and Memory-Efficient Exact Attention" (Dao et al., 2022)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Optional, Tuple
import time
import math
from dataclasses import dataclass

# For memory profiling
if torch.cuda.is_available():
    import torch.cuda as cuda

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. The O(n²) Memory Problem

### Historical Context

**The Standard Attention Problem (2017-2022)**:
```
Attention computation:
1. Compute attention scores: Q @ K^T → Shape [batch, heads, seq_len, seq_len]
2. Apply softmax: softmax(scores / √d_k)
3. Multiply by values: attention @ V

Problem: Step 1 creates n×n matrix!
- For n=4096: 4096² = 16M elements
- For n=8192: 8192² = 67M elements  
- For n=32768: 32K² = 1B elements!
```

**Memory Scaling**:
- Sequence length n=1024: ~4 MB per attention matrix
- Sequence length n=4096: ~64 MB per attention matrix
- With 32 heads, 80 layers (Llama 2 70B): ~160 GB just for attention matrices!

**Why This Matters**:
- Limits maximum context length
- Reduces possible batch size
- OOM (Out of Memory) errors
- Cannot scale to 32K+ tokens

In [ ]:
def calculate_attention_memory(seq_len, num_heads, num_layers, batch_size=1, dtype_bytes=2):
    """
    Calculate memory required for attention matrices.
    
    Args:
        seq_len: Sequence length
        num_heads: Number of attention heads
        num_layers: Number of transformer layers
        batch_size: Batch size
        dtype_bytes: Bytes per element (2 for FP16, 4 for FP32)
    
    Returns:
        Dictionary with memory breakdowns
    """
    # Attention matrix: [batch, heads, seq_len, seq_len]
    attention_matrix_size = batch_size * num_heads * seq_len * seq_len * dtype_bytes
    
    # Need one per layer
    total_attention = attention_matrix_size * num_layers
    
    return {
        'attention_matrix_mb': attention_matrix_size / 1e6,
        'attention_matrix_gb': attention_matrix_size / 1e9,
        'total_attention_gb': total_attention / 1e9,
        'per_layer_mb': attention_matrix_size / 1e6,
    }

# Analyze different models
print("Standard Attention Memory Requirements")
print("=" * 100)

configs = [
    ('GPT-2', 1024, 12, 12),
    ('Llama 2 7B', 4096, 32, 32),
    ('Llama 2 13B', 4096, 40, 40),
    ('Llama 2 70B', 4096, 64, 80),
    ('Llama 2 70B (8K ctx)', 8192, 64, 80),
    ('Llama 2 70B (32K ctx)', 32768, 64, 80),
]

results = []
for name, seq_len, num_heads, num_layers in configs:
    mem = calculate_attention_memory(seq_len, num_heads, num_layers, batch_size=1)
    results.append((name, seq_len, mem['total_attention_gb']))
    print(f"{name:25} | Seq: {seq_len:6} | Attention Memory: {mem['total_attention_gb']:6.2f} GB | Per Layer: {mem['per_layer_mb']:6.1f} MB")

print("\nKey Observations:")
print("1. Memory scales O(n²) with sequence length")
print("2. Longer contexts become prohibitively expensive")
print("3. 32K context requires 100+ GB just for attention matrices!")
print("4. This is BEFORE model weights and KV cache")

In [ ]:
# Visualize memory scaling
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Memory vs sequence length (quadratic)
ax = axes[0]
seq_lengths = np.array([512, 1024, 2048, 4096, 8192, 16384, 32768])
memory_standard = []
memory_flash = []  # Linear in sequence length

for seq_len in seq_lengths:
    mem_std = calculate_attention_memory(seq_len, num_heads=32, num_layers=32)
    memory_standard.append(mem_std['total_attention_gb'])
    # Flash Attention: O(n) memory
    memory_flash.append(mem_std['total_attention_gb'] / seq_len * 1024)  # Normalized

ax.plot(seq_lengths, memory_standard, 'o-', linewidth=2, markersize=8, 
        color='#e74c3c', label='Standard Attention O(n²)')
ax.plot(seq_lengths, memory_flash, 's-', linewidth=2, markersize=8,
        color='#2ecc71', label='Flash Attention O(n)')
ax.set_xlabel('Sequence Length', fontsize=12)
ax.set_ylabel('Memory (GB)', fontsize=12)
ax.set_title('Attention Memory Scaling (32 heads, 32 layers)', fontsize=13, fontweight='bold')
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Memory reduction factor
ax = axes[1]
reduction_factors = [std / flash for std, flash in zip(memory_standard, memory_flash)]
ax.bar(range(len(seq_lengths)), reduction_factors, color='#3498db', alpha=0.7)
ax.set_xticks(range(len(seq_lengths)))
ax.set_xticklabels([f'{s//1024}K' for s in seq_lengths])
ax.set_xlabel('Sequence Length', fontsize=12)
ax.set_ylabel('Memory Reduction Factor', fontsize=12)
ax.set_title('Flash Attention Memory Savings', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for i, val in enumerate(reduction_factors):
    ax.text(i, val, f'{val:.1f}x', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nMemory reduction at 32K context: {reduction_factors[-1]:.1f}x")
print(f"Enables: {reduction_factors[-1]:.0f}x longer contexts or {reduction_factors[-1]:.0f}x larger batches!")

## 2. GPU Memory Hierarchy

### Understanding SRAM vs HBM

**GPU Memory Architecture**:
```
SRAM (On-chip memory):
- Size: ~20-40 MB per SM (streaming multiprocessor)
- Bandwidth: ~19 TB/s (A100)
- Latency: ~5 nanoseconds
- Location: On GPU chip
- Use: Shared memory, L1 cache, registers

HBM (High Bandwidth Memory):
- Size: 40-80 GB (A100)
- Bandwidth: ~2 TB/s (A100)
- Latency: ~100-200 nanoseconds
- Location: Off-chip, adjacent to GPU
- Use: Model weights, activations, KV cache

Ratio: SRAM is ~10x faster than HBM!
```

**Standard Attention Problem**:
```python
# Standard attention must store attention matrix in HBM
scores = Q @ K.T  # Write n×n matrix to HBM (SLOW)
attn = softmax(scores)  # Read from HBM, write back (SLOW)
output = attn @ V  # Read from HBM again (SLOW)

Problem: 3 round trips to slow HBM for n×n matrix
```

**Flash Attention Solution**:
```python
# Flash Attention: compute in SRAM tiles
for block in tiles:
    # Load Q, K, V tile into SRAM
    # Compute attention entirely in SRAM (FAST)
    # Write only final output to HBM
    
Result: Only O(n) HBM accesses instead of O(n²)
```

In [ ]:
# Simulate memory access patterns
def analyze_memory_access(seq_len, block_size=64):
    """
    Analyze memory access patterns for standard vs Flash Attention.
    """
    # Standard attention
    # 1. Write attention matrix to HBM: n²
    # 2. Read for softmax: n²
    # 3. Write softmax result: n²
    # 4. Read for multiply with V: n²
    standard_hbm_access = seq_len ** 2 * 4  # 4 operations
    
    # Flash Attention
    # Only load Q, K, V (O(n)) and write output (O(n))
    num_blocks = seq_len // block_size
    flash_hbm_access = seq_len * 4  # Load Q, K, V, write output
    
    return {
        'standard_accesses': standard_hbm_access,
        'flash_accesses': flash_hbm_access,
        'reduction_factor': standard_hbm_access / flash_hbm_access,
        'num_blocks': num_blocks
    }

print("Memory Access Pattern Analysis")
print("=" * 80)

for seq_len in [1024, 2048, 4096, 8192, 16384]:
    result = analyze_memory_access(seq_len)
    print(f"Seq Length {seq_len:6}:")
    print(f"  Standard HBM accesses: {result['standard_accesses']:12,}")
    print(f"  Flash HBM accesses:    {result['flash_accesses']:12,}")
    print(f"  Reduction factor:      {result['reduction_factor']:12.1f}x")
    print()

print("Key Insight: Flash Attention reduces HBM accesses by ~1000x for long sequences!")

In [ ]:
# Visualize GPU memory hierarchy
fig, ax = plt.subplots(figsize=(12, 8))

# Memory hierarchy levels
levels = ['Registers', 'L1 Cache', 'Shared Memory\n(SRAM)', 'L2 Cache', 'HBM\n(DRAM)']
sizes_kb = [256, 128, 164, 40960, 81920000]  # A100 specs
bandwidth_gb = [19000, 19000, 19000, 7000, 2000]  # Approximate
latency_ns = [1, 5, 5, 50, 200]  # Approximate

# Create visualization
y_pos = np.arange(len(levels))
colors = ['#e74c3c', '#e67e22', '#f39c12', '#3498db', '#2ecc71']

# Size bars
ax_size = ax.twiny()
sizes_mb = [s / 1024 for s in sizes_kb]
bars = ax.barh(y_pos, [math.log10(s) for s in sizes_mb], color=colors, alpha=0.6)

# Labels
ax.set_yticks(y_pos)
ax.set_yticklabels(levels, fontsize=11)
ax.set_xlabel('log₁₀(Size in MB)', fontsize=12)
ax.set_title('GPU Memory Hierarchy (NVIDIA A100)', fontsize=14, fontweight='bold')

# Add annotations
for i, (level, size_mb, bw, lat) in enumerate(zip(levels, sizes_mb, bandwidth_gb, latency_ns)):
    # Size
    if size_mb < 1:
        size_str = f"{size_mb * 1024:.0f} KB"
    elif size_mb < 1024:
        size_str = f"{size_mb:.1f} MB"
    else:
        size_str = f"{size_mb / 1024:.1f} GB"
    
    # Bandwidth
    bw_str = f"{bw / 1000:.1f} TB/s" if bw >= 1000 else f"{bw} GB/s"
    
    # Combined info
    info = f"Size: {size_str}\nBW: {bw_str}\nLatency: {lat}ns"
    ax.text(math.log10(size_mb) + 0.5, i, info, va='center', fontsize=9)

# Highlight SRAM vs HBM
ax.axhline(y=2.5, color='red', linestyle='--', linewidth=2, alpha=0.5)
ax.text(0.5, 1, 'SRAM\n(Fast, Small)', fontsize=11, fontweight='bold', 
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.3))
ax.text(0.5, 4, 'HBM\n(Slow, Large)', fontsize=11, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='orange', alpha=0.3))

plt.tight_layout()
plt.show()

print("\nFlash Attention Strategy:")
print("Keep computation in SRAM (fast) as much as possible")
print("Minimize HBM accesses (slow) by tiling")
print("Result: 10x faster memory access patterns!")

## 3. Flash Attention Algorithm

### Tiling Strategy

**Core Idea**: Break attention computation into blocks (tiles) that fit in SRAM

```python
Standard Attention:
1. Compute full Q @ K^T matrix (n×n) → write to HBM
2. Softmax over full matrix → read from HBM, write back
3. Multiply by V → read from HBM
Problem: n² matrix doesn't fit in SRAM

Flash Attention:
1. Split Q, K, V into blocks (e.g., 64×64)
2. For each block:
   a. Load blocks into SRAM
   b. Compute attention in SRAM
   c. Incrementally update output
   d. Use online softmax (no need to store full matrix)
3. Combine block results
```

**Online Softmax**: Key innovation
```
Standard: softmax(x) requires two passes
  Pass 1: max_val = max(x)
  Pass 2: exp(x - max_val) / sum(exp(x - max_val))

Online softmax: Update in single pass
  Keep running max and sum
  Update incrementally as blocks arrive
  No need to store full attention matrix!
```

In [ ]:
def standard_attention(Q, K, V, mask=None):
    """
    Standard attention implementation.
    
    Memory: O(n²) for attention matrix
    """
    d_k = Q.size(-1)
    
    # Compute attention scores: [batch, heads, seq_len, seq_len]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Softmax: requires full matrix
    attn = F.softmax(scores, dim=-1)
    
    # Multiply by values
    output = torch.matmul(attn, V)
    
    return output, attn


def flash_attention_simplified(Q, K, V, block_size=64):
    """
    Simplified Flash Attention implementation (for illustration).
    
    Memory: O(n) instead of O(n²)
    
    Note: This is a simplified version. Real implementation uses
    custom CUDA kernels for optimal performance.
    """
    batch_size, num_heads, seq_len, d_k = Q.shape
    
    # Output accumulator
    output = torch.zeros_like(Q)
    
    # Online softmax statistics
    max_vals = torch.full((batch_size, num_heads, seq_len), float('-inf'), 
                          device=Q.device, dtype=Q.dtype)
    sum_exp = torch.zeros((batch_size, num_heads, seq_len), 
                          device=Q.device, dtype=Q.dtype)
    
    scale = 1.0 / math.sqrt(d_k)
    
    # Process in blocks (tiles)
    num_blocks = (seq_len + block_size - 1) // block_size
    
    for i in range(num_blocks):
        # Query block
        q_start = i * block_size
        q_end = min((i + 1) * block_size, seq_len)
        Q_block = Q[:, :, q_start:q_end, :]  # [B, H, block_size, d_k]
        
        for j in range(num_blocks):
            # Key/Value block
            k_start = j * block_size
            k_end = min((j + 1) * block_size, seq_len)
            K_block = K[:, :, k_start:k_end, :]  # [B, H, block_size, d_k]
            V_block = V[:, :, k_start:k_end, :]  # [B, H, block_size, d_k]
            
            # Compute attention scores for this block
            scores = torch.matmul(Q_block, K_block.transpose(-2, -1)) * scale
            
            # Online softmax update
            block_max = torch.max(scores, dim=-1, keepdim=True)[0]
            scores_exp = torch.exp(scores - block_max)
            
            # Update running statistics
            old_max = max_vals[:, :, q_start:q_end].unsqueeze(-1)
            new_max = torch.maximum(old_max, block_max)
            
            # Rescale previous sums
            correction = torch.exp(old_max - new_max)
            sum_exp[:, :, q_start:q_end] = (
                sum_exp[:, :, q_start:q_end].unsqueeze(-1) * correction +
                scores_exp.sum(dim=-1, keepdim=True)
            ).squeeze(-1)
            
            max_vals[:, :, q_start:q_end] = new_max.squeeze(-1)
            
            # Update output (weighted by unnormalized attention)
            output[:, :, q_start:q_end, :] = (
                output[:, :, q_start:q_end, :] * correction +
                torch.matmul(scores_exp, V_block)
            )
    
    # Final normalization
    output = output / sum_exp.unsqueeze(-1)
    
    return output


print("Flash Attention Implementation")
print("=" * 70)
print("\nKey Differences:")
print("1. Standard: Materializes full n×n attention matrix")
print("2. Flash: Processes blocks incrementally in SRAM")
print("3. Standard: Two-pass softmax")
print("4. Flash: Online softmax (single pass)")
print("\nResult: O(n) memory instead of O(n²)")

In [ ]:
# Test correctness: Flash Attention should match standard attention
torch.manual_seed(42)

batch_size = 2
num_heads = 4
seq_len = 256
d_k = 64

Q = torch.randn(batch_size, num_heads, seq_len, d_k, device=device)
K = torch.randn(batch_size, num_heads, seq_len, d_k, device=device)
V = torch.randn(batch_size, num_heads, seq_len, d_k, device=device)

print("Testing Flash Attention Correctness")
print("=" * 70)

# Standard attention
standard_output, _ = standard_attention(Q, K, V)

# Flash attention
flash_output = flash_attention_simplified(Q, K, V, block_size=64)

# Compare
max_diff = torch.max(torch.abs(standard_output - flash_output)).item()
mean_diff = torch.mean(torch.abs(standard_output - flash_output)).item()
relative_error = torch.mean(torch.abs(standard_output - flash_output) / 
                            (torch.abs(standard_output) + 1e-8)).item()

print(f"Max absolute difference: {max_diff:.6f}")
print(f"Mean absolute difference: {mean_diff:.6f}")
print(f"Relative error: {relative_error:.6f}")

if max_diff < 1e-4:
    print("\n✓ Flash Attention produces identical results to standard attention!")
else:
    print("\n⚠ Warning: Numerical differences detected (expected with simplified implementation)")

## 4. Memory and Speed Benchmarks

### Real-world Performance

In [ ]:
def benchmark_attention(attention_fn, Q, K, V, num_runs=10, measure_memory=True):
    """
    Benchmark attention implementation.
    """
    if device.type == 'cuda':
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        start_mem = torch.cuda.memory_allocated()
    
    # Warmup
    for _ in range(3):
        _ = attention_fn(Q, K, V)
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    # Benchmark
    times = []
    for _ in range(num_runs):
        start_time = time.time()
        output = attention_fn(Q, K, V)
        
        if device.type == 'cuda':
            torch.cuda.synchronize()
        
        times.append(time.time() - start_time)
    
    avg_time = np.mean(times)
    std_time = np.std(times)
    
    if device.type == 'cuda' and measure_memory:
        peak_mem = torch.cuda.max_memory_allocated()
        mem_used = (peak_mem - start_mem) / 1e6  # MB
    else:
        mem_used = 0
    
    return {
        'avg_time_ms': avg_time * 1000,
        'std_time_ms': std_time * 1000,
        'memory_mb': mem_used,
    }


print("Attention Implementation Benchmarks")
print("=" * 90)

configs = [
    (512, 8, 64),
    (1024, 8, 64),
    (2048, 8, 64),
]

results = []

for seq_len, num_heads, d_k in configs:
    print(f"\nSequence Length: {seq_len}, Heads: {num_heads}, d_k: {d_k}")
    print("-" * 90)
    
    # Create inputs
    Q = torch.randn(1, num_heads, seq_len, d_k, device=device)
    K = torch.randn(1, num_heads, seq_len, d_k, device=device)
    V = torch.randn(1, num_heads, seq_len, d_k, device=device)
    
    # Benchmark standard attention
    def standard_fn(q, k, v):
        output, _ = standard_attention(q, k, v)
        return output
    
    standard_result = benchmark_attention(standard_fn, Q, K, V)
    print(f"Standard Attention:")
    print(f"  Time: {standard_result['avg_time_ms']:.2f} ± {standard_result['std_time_ms']:.2f} ms")
    if device.type == 'cuda':
        print(f"  Memory: {standard_result['memory_mb']:.1f} MB")
    
    # Benchmark flash attention (simplified)
    flash_result = benchmark_attention(
        lambda q, k, v: flash_attention_simplified(q, k, v, block_size=64),
        Q, K, V
    )
    print(f"\nFlash Attention (Simplified):")
    print(f"  Time: {flash_result['avg_time_ms']:.2f} ± {flash_result['std_time_ms']:.2f} ms")
    if device.type == 'cuda':
        print(f"  Memory: {flash_result['memory_mb']:.1f} MB")
    
    # Calculate improvements
    speedup = standard_result['avg_time_ms'] / flash_result['avg_time_ms']
    if device.type == 'cuda' and standard_result['memory_mb'] > 0:
        mem_reduction = standard_result['memory_mb'] / max(flash_result['memory_mb'], 1)
        print(f"\nImprovements:")
        print(f"  Speedup: {speedup:.2f}x")
        print(f"  Memory reduction: {mem_reduction:.2f}x")
        
        results.append({
            'seq_len': seq_len,
            'speedup': speedup,
            'mem_reduction': mem_reduction
        })

print("\n" + "=" * 90)
print("\nNote: Simplified implementation doesn't use custom CUDA kernels.")
print("Real Flash Attention (with optimized CUDA) achieves:")
print("  - 2-4x speedup")
print("  - 3-10x memory reduction")
print("  - Enables 32K+ context lengths")

In [ ]:
# Visualize benchmarks
if results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    seq_lens = [r['seq_len'] for r in results]
    speedups = [r['speedup'] for r in results]
    mem_reductions = [r['mem_reduction'] for r in results]
    
    # Speedup
    ax = axes[0]
    ax.bar(range(len(seq_lens)), speedups, color='#3498db', alpha=0.7)
    ax.set_xticks(range(len(seq_lens)))
    ax.set_xticklabels(seq_lens)
    ax.set_xlabel('Sequence Length', fontsize=12)
    ax.set_ylabel('Speedup', fontsize=12)
    ax.set_title('Flash Attention Speedup', fontsize=13, fontweight='bold')
    ax.axhline(y=1, color='red', linestyle='--', alpha=0.5, label='Baseline')
    ax.grid(True, alpha=0.3, axis='y')
    
    for i, val in enumerate(speedups):
        ax.text(i, val, f'{val:.2f}x', ha='center', va='bottom', 
                fontsize=10, fontweight='bold')
    
    # Memory reduction
    ax = axes[1]
    ax.bar(range(len(seq_lens)), mem_reductions, color='#2ecc71', alpha=0.7)
    ax.set_xticks(range(len(seq_lens)))
    ax.set_xticklabels(seq_lens)
    ax.set_xlabel('Sequence Length', fontsize=12)
    ax.set_ylabel('Memory Reduction', fontsize=12)
    ax.set_title('Flash Attention Memory Savings', fontsize=13, fontweight='bold')
    ax.axhline(y=1, color='red', linestyle='--', alpha=0.5, label='Baseline')
    ax.grid(True, alpha=0.3, axis='y')
    
    for i, val in enumerate(mem_reductions):
        ax.text(i, val, f'{val:.2f}x', ha='center', va='bottom',
                fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

## 5. Installing and Using flash-attn

### Production-Ready Flash Attention

The real Flash Attention library uses highly optimized CUDA kernels.

In [ ]:
print("Installing flash-attn")
print("=" * 70)
print("""
# Installation (requires CUDA and PyTorch)
pip install flash-attn --no-build-isolation

# Or from source:
git clone https://github.com/Dao-AILab/flash-attention
cd flash-attention
python setup.py install

Requirements:
- CUDA 11.6+
- PyTorch 1.12+
- GPU: Ampere (A100), Ada (RTX 4090), or newer
- For older GPUs: Use xformers or PyTorch's scaled_dot_product_attention
""")

# Try to import flash attention
try:
    from flash_attn import flash_attn_func
    flash_attn_available = True
    print("✓ flash-attn is installed and available!")
except ImportError:
    flash_attn_available = False
    print("✗ flash-attn not installed (using simplified version for demo)")
    print("  Install with: pip install flash-attn --no-build-isolation")

In [ ]:
# Usage example with flash-attn
if flash_attn_available:
    print("\nUsing Real Flash Attention")
    print("=" * 70)
    
    from flash_attn import flash_attn_func
    
    # Flash attention expects specific input format:
    # Q, K, V: [batch, seq_len, num_heads, head_dim]
    batch_size = 2
    seq_len = 2048
    num_heads = 32
    head_dim = 128
    
    q = torch.randn(batch_size, seq_len, num_heads, head_dim, device=device, dtype=torch.float16)
    k = torch.randn(batch_size, seq_len, num_heads, head_dim, device=device, dtype=torch.float16)
    v = torch.randn(batch_size, seq_len, num_heads, head_dim, device=device, dtype=torch.float16)
    
    # Run flash attention
    output = flash_attn_func(q, k, v, dropout_p=0.0, causal=False)
    
    print(f"Input shape: {q.shape}")
    print(f"Output shape: {output.shape}")
    print("\n✓ Flash Attention executed successfully!")
    
    # Benchmark
    print("\nBenchmarking Real Flash Attention...")
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(100):
        output = flash_attn_func(q, k, v, dropout_p=0.0, causal=False)
    torch.cuda.synchronize()
    avg_time = (time.time() - start) / 100
    
    print(f"Average time: {avg_time * 1000:.2f} ms")
    print(f"Throughput: {batch_size * seq_len / avg_time:.0f} tokens/sec")

else:
    print("\nFlash Attention Library Usage (Example)")
    print("=" * 70)
    print("""
from flash_attn import flash_attn_func

# Inputs: [batch, seq_len, num_heads, head_dim]
q = torch.randn(batch, seqlen, nheads, headdim, device='cuda', dtype=torch.float16)
k = torch.randn(batch, seqlen, nheads, headdim, device='cuda', dtype=torch.float16)
v = torch.randn(batch, seqlen, nheads, headdim, device='cuda', dtype=torch.float16)

# Run flash attention
output = flash_attn_func(q, k, v, dropout_p=0.0, causal=False)

# For causal (decoder) attention:
output = flash_attn_func(q, k, v, dropout_p=0.0, causal=True)

# Returns: [batch, seq_len, num_heads, head_dim]
    """)

## 6. Integration with Models

### Using Flash Attention in Transformers

In [ ]:
print("Integrating Flash Attention with Models")
print("=" * 70)

integration_guide = """
1. HUGGING FACE TRANSFORMERS:

from transformers import AutoModelForCausalLM

# Many models support Flash Attention v2 natively
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="flash_attention_2"  # Use Flash Attention
)

# Supported models (as of 2024):
# - Llama, Llama 2, Llama 3
# - Mistral, Mixtral
# - GPT-NeoX
# - Falcon
# - And many more...


2. PYTORCH SCALED_DOT_PRODUCT_ATTENTION:

# PyTorch 2.0+ has built-in memory-efficient attention
import torch.nn.functional as F

# Automatically uses Flash Attention if available
output = F.scaled_dot_product_attention(
    q, k, v,
    attn_mask=None,
    dropout_p=0.0,
    is_causal=True  # For decoder models
)

# PyTorch automatically selects best implementation:
# 1. Flash Attention (if available)
# 2. Memory-efficient attention (xformers)
# 3. Standard attention (fallback)


3. CUSTOM ATTENTION MODULE:

class FlashMultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.head_dim = d_model // num_heads
        
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
    
    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        
        # Project to Q, K, V
        qkv = self.qkv_proj(x)  # [B, L, 3*d_model]
        qkv = qkv.reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 1, 3, 4)  # [3, B, L, H, d_k]
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        # Use Flash Attention
        if flash_attn_available:
            from flash_attn import flash_attn_func
            attn_output = flash_attn_func(q, k, v, dropout_p=0.0, causal=True)
        else:
            # Fallback to PyTorch scaled dot product
            q = q.transpose(1, 2)  # [B, H, L, d_k]
            k = k.transpose(1, 2)
            v = v.transpose(1, 2)
            attn_output = F.scaled_dot_product_attention(q, k, v, is_causal=True)
            attn_output = attn_output.transpose(1, 2)  # [B, L, H, d_k]
        
        # Reshape and project
        attn_output = attn_output.reshape(batch_size, seq_len, self.d_model)
        output = self.out_proj(attn_output)
        
        return output


4. VLLM (Production Serving):

from vllm import LLM

# vLLM automatically uses Flash Attention
llm = LLM(
    model="meta-llama/Llama-2-7b-hf",
    dtype="float16",
    max_model_len=8192,  # Long context possible with Flash Attention
)

# No extra configuration needed - Flash Attention is default!
"""

print(integration_guide)

In [ ]:
# Demonstrate PyTorch's scaled_dot_product_attention
print("\nUsing PyTorch 2.0+ Scaled Dot Product Attention")
print("=" * 70)

if hasattr(F, 'scaled_dot_product_attention'):
    batch_size = 2
    num_heads = 8
    seq_len = 1024
    head_dim = 64
    
    q = torch.randn(batch_size, num_heads, seq_len, head_dim, device=device)
    k = torch.randn(batch_size, num_heads, seq_len, head_dim, device=device)
    v = torch.randn(batch_size, num_heads, seq_len, head_dim, device=device)
    
    # PyTorch automatically selects best implementation
    output = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    
    print(f"Input shape: {q.shape}")
    print(f"Output shape: {output.shape}")
    print("\n✓ PyTorch automatically used memory-efficient attention!")
    print("  (Flash Attention if available, else memory-efficient fallback)")
else:
    print("PyTorch 2.0+ required for scaled_dot_product_attention")
    print("Upgrade with: pip install --upgrade torch")

## 7. Flash Attention v2 and v3

### Evolution of Flash Attention

In [ ]:
print("Flash Attention Evolution")
print("=" * 90)

versions = [
    {
        'version': 'Flash Attention v1',
        'year': '2022',
        'speedup': '2-3x',
        'memory': 'O(n) instead of O(n²)',
        'key_innovation': 'Tiling + online softmax',
        'limitations': 'Not fully optimized for all GPUs',
    },
    {
        'version': 'Flash Attention v2',
        'year': '2023',
        'speedup': '2-4x (2x faster than v1)',
        'memory': 'Same O(n) as v1',
        'key_innovation': 'Better parallelization, reduced non-matmul FLOPs',
        'limitations': 'Requires Ampere or newer GPUs',
    },
    {
        'version': 'Flash Attention v3',
        'year': '2024',
        'speedup': '1.5-2x faster than v2',
        'memory': 'Same O(n) as v1/v2',
        'key_innovation': 'Hardware-specific optimization for H100, asynchronous operations',
        'limitations': 'H100 only (for full benefits)',
    },
]

for v in versions:
    print(f"\n{v['version']} ({v['year']}):")
    print(f"  Speedup: {v['speedup']}")
    print(f"  Memory: {v['memory']}")
    print(f"  Innovation: {v['key_innovation']}")
    print(f"  Limitations: {v['limitations']}")

print("\n" + "=" * 90)
print("\nRecommendations:")
print("- Ampere GPUs (A100, RTX 3090): Use Flash Attention v2")
print("- Ada/Hopper GPUs (RTX 4090, H100): Use Flash Attention v2 or v3")
print("- Older GPUs: Use xformers or PyTorch's memory-efficient attention")
print("- Production: vLLM or TensorRT-LLM (includes Flash Attention automatically)")

In [ ]:
# Visualize improvements over time
fig, ax = plt.subplots(figsize=(12, 6))

versions_names = ['Standard\nAttention', 'Flash\nAttention v1', 'Flash\nAttention v2', 'Flash\nAttention v3']
speedups = [1.0, 2.5, 4.0, 6.0]  # Relative to standard
memory_usage = [100, 30, 30, 30]  # Percentage relative to standard

x = np.arange(len(versions_names))
width = 0.35

bars1 = ax.bar(x - width/2, speedups, width, label='Speedup', color='#3498db', alpha=0.7)
bars2 = ax.bar(x + width/2, [m/20 for m in memory_usage], width, 
               label='Memory (÷20)', color='#e74c3c', alpha=0.7)

ax.set_ylabel('Relative Performance', fontsize=12)
ax.set_title('Flash Attention Evolution (2022-2024)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(versions_names)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, val in zip(bars1, speedups):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.1f}x', ha='center', va='bottom', fontsize=10, fontweight='bold')

for bar, val in zip(bars2, memory_usage):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nKey Takeaway:")
print("Flash Attention v1: Memory breakthrough (O(n²) → O(n))")
print("Flash Attention v2: Speed optimization (2x faster)")
print("Flash Attention v3: Hardware specialization (H100)")

## Summary

### Key Takeaways

1. **The Problem**:
   - Standard attention requires O(n²) memory for attention matrix
   - Limits maximum context length to 2-4K tokens
   - Bottleneck: Storing n×n matrix in slow HBM memory

2. **Flash Attention Solution**:
   - Tile computation into blocks that fit in fast SRAM
   - Online softmax: avoid materializing full attention matrix
   - Reduces memory from O(n²) to O(n)
   - Result: 2-4x faster, 2-4x less memory

3. **Performance Gains**:
   - Memory: 2-4x reduction (enables longer contexts)
   - Speed: 1.5-2x faster (v1), 2-4x (v2), up to 6x (v3 on H100)
   - Enables: 32K+ token contexts, larger batch sizes

4. **GPU Memory Hierarchy**:
   - SRAM (on-chip): Fast (19 TB/s) but small (20-40 MB)
   - HBM (off-chip): Slower (2 TB/s) but large (80 GB)
   - Key: Keep computation in SRAM, minimize HBM access

5. **Integration**:
   - Hugging Face: `attn_implementation="flash_attention_2"`
   - PyTorch 2.0+: `F.scaled_dot_product_attention()` (automatic)
   - vLLM: Flash Attention enabled by default
   - Custom: Use `flash_attn` library directly

6. **Hardware Requirements**:
   - Ampere+ (A100, RTX 3090): Full Flash Attention v2 support
   - Ada/Hopper (RTX 4090, H100): Flash Attention v3 available
   - Older GPUs: Use xformers or PyTorch fallback

### Historical Impact

- **2022**: Flash Attention v1 paper (Dao et al.) - Memory breakthrough
- **2023**: Flash Attention v2 - 2x speed improvement
- **2023**: Rapid adoption in all major frameworks
- **2024**: Flash Attention v3 - Hardware-specific optimization
- **Impact**: Enabled 32K+ context models, made long-context inference practical

### When to Use

**Always use Flash Attention when**:
- ✓ Sequence length > 1024 tokens
- ✓ Need to maximize batch size
- ✓ Long context inference (8K+)
- ✓ Memory-constrained serving
- ✓ Using compatible GPU (Ampere+)

**Standard attention acceptable when**:
- Short sequences (<512 tokens)
- Older GPUs without Flash Attention support
- Prototyping/debugging (simpler code)

### Next Steps

- **Notebook 21**: MQA/GQA Implementation (reduce KV cache 4-32x)
- **Notebook 23**: Long Context Inference (32K+ tokens)
- **Notebook 24**: Memory Optimization Comparison (all techniques)
- **Notebook 55**: KV Cache Optimization (quantization, compression)

### Further Reading

- Paper: "FlashAttention: Fast and Memory-Efficient Exact Attention" (Dao et al., 2022)
- Paper: "FlashAttention-2: Faster Attention with Better Parallelism" (Dao, 2023)
- GitHub: https://github.com/Dao-AILab/flash-attention
- LLM_INFERENCE_ROADMAP.md - Stage 2: Memory Optimization